In [ ]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

# Read all telemetry log files and append into one dataframe
telemetry_files = sorted(Path("logs").glob("telemetry_*.csv"))
df = pd.concat(
    (pd.read_csv(path, parse_dates=["ts"]) for path in telemetry_files),
    ignore_index=True,
)
df = df.sort_values("ts", ignore_index=True)
df


In [ ]:
# Add a column for anomaly annotation
df["is_temp_anomaly"] = False
df["is_hum_anomaly"] = False

# Manual anomaly annotations (set time intervals where anomalies are)
temp_anomaly_windows = [
    # Z-score time window
    {"start": "2026-01-27 16:08:00", "end": "2026-01-27 16:09:40"},
    {"start": "2026-01-27 16:24:08", "end": "2026-01-27 16:25:28"},
    {"start": "2026-01-27 16:32:04", "end": "2026-01-27 16:34:58"},
    {"start": "2026-01-27 16:40:02", "end": "2026-01-27 16:43:54"},
    {"start": "2026-01-27 16:48:02", "end": "2026-01-27 16:51:54"},
    {"start": "2026-01-27 16:56:04", "end": "2026-01-27 17:00:00"},
    # ewma time window
    {"start": "2026-01-28 08:16:20", "end": "2026-01-28 08:20:06"},
    {"start": "2026-01-28 08:24:20", "end": "2026-01-28 08:28:34"},
    {"start": "2026-01-28 08:40:20", "end": "2026-01-28 08:45:36"},
    {"start": "2026-01-28 08:48:22", "end": "2026-01-28 08:53:50"},
    {"start": "2026-01-28 08:56:20", "end": "2026-01-28 09:00:00"},
    # rulebased time window
    {"start": "2026-01-28 09:08:08", "end": "2026-01-28 09:09:30"},
    {"start": "2026-01-28 09:16:04", "end": "2026-01-28 09:18:04"},
    {"start": "2026-01-28 09:24:12", "end": "2026-01-28 09:26:40"},
    {"start": "2026-01-28 09:40:08", "end": "2026-01-28 09:43:56"},
    {"start": "2026-01-28 09:48:10", "end": "2026-01-28 09:54:26"},
    {"start": "2026-01-28 09:56:10", "end": "2026-01-28 10:00:00"},
]

hum_anomaly_windows = [
    # z-score time window
    {"start": "2026-01-27 16:08:02", "end": "2026-01-27 16:08:48"},
    {"start": "2026-01-27 16:16:02", "end": "2026-01-27 16:16:58"},
    {"start": "2026-01-27 16:24:02", "end": "2026-01-27 16:24:58"},
    {"start": "2026-01-27 16:32:00", "end": "2026-01-27 16:33:20"},
    {"start": "2026-01-27 16:40:04", "end": "2026-01-27 16:44:00"},
    {"start": "2026-01-27 16:48:04", "end": "2026-01-27 16:52:02"},
    {"start": "2026-01-27 16:56:04", "end": "2026-01-27 17:00:00"},
    # ewma time window
    {"start": "2026-01-28 08:08:16", "end": "2026-01-28 08:09:00"},
    {"start": "2026-01-28 08:16:20", "end": "2026-01-28 08:17:00"},
    {"start": "2026-01-28 08:24:20", "end": "2026-01-28 08:25:42"},
    {"start": "2026-01-28 08:32:04", "end": "2026-01-28 08:33:46"},
    {"start": "2026-01-28 08:40:22", "end": "2026-01-28 08:43:34"},
    {"start": "2026-01-28 08:48:22", "end": "2026-01-28 08:53:10"},
    {"start": "2026-01-28 08:56:22", "end": "2026-01-28 09:00:00"},
    # rulebased time window
    {"start": "2026-01-28 09:08:06", "end": "2026-01-28 09:08:32"},
    {"start": "2026-01-28 09:16:08", "end": "2026-01-28 09:16:44"},
    {"start": "2026-01-28 09:24:10", "end": "2026-01-28 09:24:50"},
    {"start": "2026-01-28 09:29:44", "end": "2026-01-28 09:30:40"},
    {"start": "2026-01-28 09:32:10", "end": "2026-01-28 09:33:04"},
    {"start": "2026-01-28 09:40:10", "end": "2026-01-28 09:44:02"},
    {"start": "2026-01-28 09:48:10", "end": "2026-01-28 09:54:34"},
    {"start": "2026-01-28 09:56:10", "end": "2026-01-28 10:00:00"},
]


def apply_time_windows(df, windows, col, ts_col="ts"):
    if not windows:
        return
    for window in windows:
        start = pd.to_datetime(window["start"])
        end = pd.to_datetime(window["end"])
        mask = (df[ts_col] >= start) & (df[ts_col] <= end)
        df.loc[mask, col] = True


apply_time_windows(df, temp_anomaly_windows, "is_temp_anomaly")
apply_time_windows(df, hum_anomaly_windows, "is_hum_anomaly")

df


In [ ]:
import math

# Define detector windows and corresponding columns for predictions
COMPUTE_CLASSIC_METRICS = False
detector_windows = [
    {
        "name": "zscore",
        "start": "2026-01-27 16:00",
        "end": "2026-01-27 17:00",
        "temp_flag": "temp_zscore_anomaly",
        "hum_flag": "hum_zscore_anomaly",
    },
    {
        "name": "ewma",
        "start": "2026-01-28 08:00",
        "end": "2026-01-28 09:00",
        "temp_flag": "temp_ewma_anomaly",
        "hum_flag": "hum_ewma_anomaly",
    },
    {
        "name": "rulebased",
        "start": "2026-01-28 09:01",
        "end": "2026-01-28 10:00",
        "temp_flag": "temp_rulebased_anomaly",
        "hum_flag": "hum_rulebased_anomaly",
    },
    {
        "name": "monitoring-with-mqtt",
        "start": "2026-01-25 16:00",
        "end": "2026-01-25 17:00",
        "temp_flag": "",
        "hum_flag": "",
    },
    {
        "name": "monitoring-baseline",
        "start": "2000-01-01 00:00",
        "end": "2000-01-01 01:00",
        "temp_flag": "",
        "hum_flag": "",
    },
    {
        "name": "monitoring-with-conn",
        "start": "2026-01-25 20:00",
        "end": "2026-01-25 21:00",
        "temp_flag": "",
        "hum_flag": "",
    },
]

# Classic evaluation metrics computation
def _compute_metrics(y_true, y_pred):
    tp = int((y_true & y_pred).sum())
    fp = int((~y_true & y_pred).sum())
    fn = int((y_true & ~y_pred).sum())
    tn = int((~y_true & ~y_pred).sum())
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    if math.isnan(precision) or math.isnan(recall) or (precision + recall) == 0:
        f1 = float("nan")
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

# Event-based evaluation for anomaly detection (strict interval matching).
def _cluster_alarm_events(detections, max_gap_seconds=8):
    """Cluster detections into alarm events using a max gap of 8 seconds."""
    if not detections:
        return []

    times = [ts for ts in detections]
    times.sort()

    max_gap = pd.Timedelta(seconds=max_gap_seconds)
    alarm_events = [times[0]]
    prev_ts = times[0]
    for ts in times[1:]:
        if ts - prev_ts > max_gap:
            alarm_events.append(ts)
        prev_ts = ts
    return alarm_events
def evaluate_event_f1(ground_truth_intervals, detections):
    """Compute event-based precision/recall/F1 with strict interval matching."""
    alarm_events = _cluster_alarm_events(detections)

    tp = 0
    fn = 0
    for start_ts, end_ts in ground_truth_intervals:
        matched = False
        for alarm_ts in alarm_events:
            if alarm_ts < start_ts:
                continue
            if alarm_ts > end_ts:
                break
            matched = True
            break
        if matched:
            tp += 1
        else:
            fn += 1

    fp = 0
    interval_idx = 0
    for alarm_ts in alarm_events:
        while interval_idx < len(ground_truth_intervals) and alarm_ts > ground_truth_intervals[interval_idx][1]:
            interval_idx += 1
        if interval_idx < len(ground_truth_intervals):
            start_ts, end_ts = ground_truth_intervals[interval_idx]
            if start_ts <= alarm_ts <= end_ts:
                continue
        fp += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "merged_intervals": ground_truth_intervals,
        "alarm_events": alarm_events,
    }

def _intervals_from_series(ts_series, flag_series):
    """Convert boolean series into inclusive [start, end] intervals."""
    ts_series = pd.to_datetime(ts_series)
    intervals = []
    start_ts = None
    end_ts = None
    for ts, flag in zip(ts_series, flag_series):
        if flag:
            if start_ts is None:
                start_ts = ts
            end_ts = ts
        elif start_ts is not None:
            intervals.append((start_ts, end_ts))
            start_ts = None
            end_ts = None
    if start_ts is not None:
        intervals.append((start_ts, end_ts))
    return intervals

rows = []
classic_rows = []
window_rows = []
for window in detector_windows:
    start = pd.to_datetime(window["start"])
    end = pd.to_datetime(window["end"])
    mask = (df["ts"] >= start) & (df["ts"] <= end)

    # calculate average resource usage metrics
    # avg mp_cpu
    mp_cpu = pd.to_numeric(
        df.loc[mask, "mp_cpu"], errors="coerce"
    )
    avg_mp_cpu = mp_cpu.mean()
    std_mp_cpu = mp_cpu.std(ddof=0)
    # avg cpu_core0
    cpu_core0 = pd.to_numeric(
        df.loc[mask, "cpu_core0"], errors="coerce"
    )
    avg_cpu_core0 = cpu_core0.mean()
    std_cpu_core0 = cpu_core0.std(ddof=0)
    # avg cpu_core1
    cpu_core1 = pd.to_numeric(
        df.loc[mask, "cpu_core1"], errors="coerce"
    )
    avg_cpu_core1 = cpu_core1.mean()
    std_cpu_core1 = cpu_core1.std(ddof=0)
    # avg mp_mem_ratio,
    used = pd.to_numeric(df.loc[mask, "mp_used_kb"], errors="coerce")
    total = pd.to_numeric(df.loc[mask, "mp_total_kb"], errors="coerce")
    mem_ratio = used / total * 100
    mem_ratio = mem_ratio.replace([float("inf"), -float("inf")], pd.NA)
    avg_mp_mem_ratio = mem_ratio.mean()
    std_mp_mem_ratio = mem_ratio.std(ddof=0)
    # avg idf_mem_ratio
    used = pd.to_numeric(df.loc[mask, "idf_used_kb"], errors="coerce")
    total = pd.to_numeric(df.loc[mask, "idf_total_kb"], errors="coerce")
    mem_ratio = used / total * 100
    mem_ratio = mem_ratio.replace([float("inf"), -float("inf")], pd.NA)
    avg_idf_mem_ratio = mem_ratio.mean()
    std_idf_mem_ratio = mem_ratio.std(ddof=0)

    window_rows.append(
        {
            "method": window["name"],
            "start": start,
            "end": end,
            "avg_mp_cpu": avg_mp_cpu,
            "std_mp_cpu": std_mp_cpu,
            "avg_cpu_core0": avg_cpu_core0,
            "std_cpu_core0": std_cpu_core0,
            "avg_cpu_core1": avg_cpu_core1,
            "std_cpu_core1": std_cpu_core1,
            "avg_mp_mem_ratio": avg_mp_mem_ratio,
            "std_mp_mem_ratio": std_mp_mem_ratio,
            "avg_idf_mem_ratio": avg_idf_mem_ratio,
            "std_idf_mem_ratio": std_idf_mem_ratio,
        }
    )

    # calculate evaluation metrics
    for metric_name, pred_col, true_col in [
        ("temp", window["temp_flag"], "is_temp_anomaly"),
        ("hum", window["hum_flag"], "is_hum_anomaly"),
    ]:
        if pred_col not in df.columns or true_col not in df.columns:
            continue
        window_df = df.loc[mask, ["ts", pred_col, true_col]].copy()
        window_df = window_df.sort_values("ts")
        ground_truth_intervals = _intervals_from_series(
            window_df["ts"], window_df[true_col]
        )
        detections_ts = window_df.loc[window_df[pred_col], "ts"].tolist()
        stats = evaluate_event_f1(ground_truth_intervals, detections_ts)
        rows.append(
            {
                "method": window["name"],
                "metric": metric_name,
                "start": start,
                "end": end,
                **stats,
            }
        )

        if COMPUTE_CLASSIC_METRICS:
            classic_stats = _compute_metrics(
                window_df[true_col], window_df[pred_col]
            )
            classic_rows.append(
                {
                    "method": window["name"],
                    "metric": metric_name,
                    "start": start,
                    "end": end,
                    **classic_stats,
                }
            )

metrics_df = pd.DataFrame(rows)
if not metrics_df.empty:
    metrics_df = metrics_df.sort_values(["method", "metric", "start"])

classic_metrics_df = pd.DataFrame(classic_rows)
if not classic_metrics_df.empty:
    classic_metrics_df = classic_metrics_df.sort_values(["method", "metric", "start"])

resource_df = pd.DataFrame(window_rows)
if not resource_df.empty:
    resource_df = resource_df.sort_values(["method", "start"])

# Display dataframe with evaluation metrics
metrics_df


In [ ]:
# Classic point-wise metrics (optional)
if COMPUTE_CLASSIC_METRICS and not classic_metrics_df.empty:
    display(classic_metrics_df)

In [ ]:
# Display second dataframe with resource usage
resource_df

In [ ]:
# Bar chart for average CPU + memory ratios
if resource_df.empty:
    print('resource_df is empty; no data to plot.')
else:
    plot_df = resource_df.copy()
    plot_df['method'] = plot_df['method'].astype(str)

    metrics = [
        ('avg_cpu_core0', 'CPU Kern 0'),
        ('avg_cpu_core1', 'CPU Kern 1'),
        ('avg_mp_mem_ratio', 'Micropython RAM'),
        ('avg_idf_mem_ratio', 'ESP-IDF Ram'),
    ]

    def _wrap_label(text, max_len=14):
        words = text.split()
        lines = []
        current = ''
        for word in words:
            if not current:
                current = word
                continue
            if len(current) + 1 + len(word) <= max_len:
                current = f'{current} {word}'
            else:
                lines.append(current)
                current = word
        if current:
            lines.append(current)
        return '<br>'.join(lines)

    label_map = {
        'zscore': 'Z-Score',
        'ewma': 'EWMA',
        'rulebased': 'Regelbasiert',
        'monitoring-baseline': 'Monitoring Baseline',
        'monitoring-with-conn': 'Monitoring mit MQTT-Verbindung',
        'monitoring-with-mqtt': 'Monitoring mit Dauerkommunikation',
    }

    detector_order = [
        'zscore',
        'ewma',
        'rulebased',
    ]

    monitoring_order = [
        'monitoring-baseline',
        'monitoring-with-conn',
        'monitoring-with-mqtt',
    ]

    def _make_chart(order, title):
        subset_df = plot_df.set_index('method').reindex(order).reset_index()

        max_y = (
            subset_df[[m[0] for m in metrics]].values.max()
            + subset_df[[m[0].replace('avg_', 'std_') for m in metrics]].values.max()
        )
        y_padding = 0.03 * max_y if pd.notna(max_y) else 0.0

        fig = go.Figure()
        annotations = []
        n_metrics = len(metrics)
        x_shift_step = 50

        for idx, (col, label) in enumerate(metrics):
            values = subset_df[col]
            std_col = col.replace('avg_', 'std_')
            std_vals = subset_df[std_col]
            fig.add_bar(
                x=subset_df['method'],
                y=values,
                name=label,
                error_y=dict(type='data', array=std_vals, visible=True),
            )

            xshift = int((idx - (n_metrics - 1) / 2) * x_shift_step)
            for method, v, s in zip(subset_df['method'], values, std_vals):
                if pd.isna(v):
                    continue
                s_val = 0.0 if pd.isna(s) else s
                annotations.append(
                    dict(
                        x=method,
                        y=v + s_val + y_padding,
                        text=f'{v:.2f}',
                        xref='x',
                        yref='y',
                        showarrow=False,
                        xshift=xshift,
                        yanchor='bottom',
                    )
                )

        fig.update_layout(
            width=1000,
            height=420,
            title=title,
            template='plotly_white',
            xaxis=dict(
                categoryorder='array',
                categoryarray=order,
                tickvals=order,
                ticktext=[label_map.get(m, m) for m in order],
                title='Verfahren',
                tickangle=0,
            ),
            yaxis_title='Durchschnittlicher Verbrauch (%)',
            yaxis_range=[0, max_y * 1.2] if pd.notna(max_y) else None,
            barmode='group',
            legend_title='Metrik',
            uniformtext_minsize=9,
            uniformtext_mode='show',
            annotations=annotations,
        )
        fig.update_traces(cliponaxis=False)
        fig.show()

    _make_chart(detector_order, 'Durchschnittlicher Ressourcenverbrauch pro Verfahren')
    _make_chart(monitoring_order, 'Durchschnittlicher Ressourcenverbrauch bei Testläufen')


In [ ]:
# Filter out data before 2026
cutoff = pd.Timestamp("2026-01-27 16:00")
df = df.loc[df["ts"] >= cutoff].reset_index(drop=True)


In [ ]:
# Plot temperature over time
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=df["ts"], y=df["temp"], name="Temp (C)")
)

detectors = {
    "zscore": {"temp": "temp_zscore_anomaly"},
    "ewma": {"temp": "temp_ewma_anomaly"},
    "rulebased": {"temp": "temp_rulebased_anomaly"},
}
symbol_map = {"zscore": "x", "ewma": "triangle-up", "rulebased": "diamond"}
metric_colors = {"temp": "#d62728"}
legend_shown = set()

for detector, cols in detectors.items():
    for metric, col in cols.items():
        mask = df[col]
        if mask.any():
            fig.add_trace(
                go.Scatter(
                    x=df.loc[mask, "ts"],
                    y=df.loc[mask, metric],
                    mode="markers",
                    name=detector,
                    legendgroup=detector,
                    showlegend=detector not in legend_shown,
                    marker={
                        "symbol": symbol_map[detector],
                        "size": 8,
                        "color": metric_colors[metric],
                    },
                )
            )
            legend_shown.add(detector)

fig.update_layout(
    title="Temperatur Zeitreihe",
    xaxis={"title": "Zeit"},
    yaxis={"title": "Temp (C)"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    template='plotly_white',
)
fig.show()


In [ ]:
# Plot humidity over time
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=df["ts"], y=df["hum"], name="RH (%)")
)

detectors = {
    "zscore": {"hum": "hum_zscore_anomaly"},
    "ewma": {"hum": "hum_ewma_anomaly"},
    "rulebased": {"hum": "hum_rulebased_anomaly"},
}
symbol_map = {"zscore": "x", "ewma": "triangle-up", "rulebased": "diamond"}
metric_colors = {"hum": "#d62728"}
legend_shown = set()

for detector, cols in detectors.items():
    for metric, col in cols.items():
        mask = df[col]
        if mask.any():
            fig.add_trace(
                go.Scatter(
                    x=df.loc[mask, "ts"],
                    y=df.loc[mask, metric],
                    mode="markers",
                    name=detector,
                    legendgroup=detector,
                    showlegend=detector not in legend_shown,
                    marker={
                        "symbol": symbol_map[detector],
                        "size": 8,
                        "color": metric_colors[metric],
                    },
                )
            )
            legend_shown.add(detector)

fig.update_layout(
    title="Luftfeuchtigkeit Zeitreihe",
    xaxis={"title": "Zeit"},
    yaxis={"title": "RH (%)"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    template='plotly_white',
)
fig.show()
